# Librerías

In [2]:
# Manipulacion de datos
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', 200)
import json
import os
import unicodedata


from pathlib import Path

# Paths 

In [3]:
csv_path = Path('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_agricolas/')

# Funciones

In [4]:
# Función para eliminar acentos (la misma que usamos antes)
def eliminar_acentos(texto):
    if isinstance(texto, str):
        texto = unicodedata.normalize('NFD', texto)
        texto = texto.encode('ascii', 'ignore')
        texto = texto.decode("utf-8")
    return texto

tipos_de_datos = {
    'cosechada': np.float64,
    'precio': np.float64,
    'rendimiento': np.float64,
    'sembrada': np.float64,
    'siniestrada': np.float64,
    'valorproduccion': np.float64,
    'volumenproduccion': np.float64,
    # 'preciomediorural': np.float64,
    'anio': 'string',
    'idcader': 'string',
    'idciclo': 'string',
    'idcultivo': 'string',
    'idddr': 'string',
    'idestado': 'string',
    'idmodalidad': 'string',
    'idmunicipio': 'string',
    'idunidadmedida': 'string',
    'nomcader': 'string',
    'nomcicloproductivo': 'string',
    'nomcultivo': 'string',
    'nomddr': 'string',
    'nomestado': 'string',
    'nommodalidad': 'string',
    'nommunicipio': 'string',
    'nomunidad': 'string'
}

def procesar_csv(ruta_csv, encoding, tipos_de_datos):
    """
    Función para procesar un archivo CSV:
    1. Lee el CSV.
    2. Convierte los nombres de las columnas a minúsculas.
    3. Elimina acentos de los nombres de las columnas.
    4. Ordena las columnas alfabéticamente.
    5. Asigna los tipos de datos especificados.

    Parámetros:
        ruta_csv (str): Ruta del archivo CSV.
        encoding (str): Encoding del archivo CSV (ej: 'latin-1', 'utf-8').
        tipos_de_datos (dict): Diccionario con los tipos de datos para las columnas.

    Retorna:
        pd.DataFrame: DataFrame procesado.
    """
    # Leer el CSV
    df = pd.read_csv(ruta_csv, encoding=encoding, low_memory=False)

    # Convertir nombres de columnas a minúsculas
    df.columns = df.columns.str.lower()
    #print(f"Columnas después de convertir a minúsculas: {df.columns.tolist()}")

    # Eliminar acentos de los nombres de las columnas
    df.columns = [eliminar_acentos(col) for col in df.columns]

    # Ordenar columnas alfabéticamente
    df = df.reindex(sorted(df.columns), axis=1)
    df = df.rename(columns={'preciomediorural': 'precio'})

    # Reemplazar '#¡NUM!' con NaN para columnas numéricas
    for col in df.columns:
        if tipos_de_datos.get(col) in [np.float64, np.int64]:  # Check if column is numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Asignar tipos de datos
    df = df.astype(tipos_de_datos)

    return df






# Generar el csv limpio y filtrado

In [5]:
dataframes = []

# Iterar sobre todos los archivos .csv en la carpeta
for archivo in csv_path.glob('*.csv'):
    #print(archivo)
    df_temporal = procesar_csv(encoding='latin-1', ruta_csv=archivo, tipos_de_datos=tipos_de_datos)
    dataframes.append(df_temporal)

# Concatenar todos los archivos limpios en un solo DataFrame
df = pd.concat(dataframes, ignore_index=True)


In [6]:
df_final = df[(df['nomcultivo']=='Maíz grano') ]

In [7]:
df_final.shape

(55853, 24)

In [8]:
df_final

,anio,cosechada,idcader,idciclo,idcultivo,idddr,idestado,idmodalidad,idmunicipio,idunidadmedida,nomcader,nomcicloproductivo,nomcultivo,nomddr,nomestado,nommodalidad,nommunicipio,nomunidad,precio,rendimiento,sembrada,siniestrada,valorproduccion,volumenproduccion
7,2016,300.0,1,2,7490000,1,1,1,1,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Riego,Aguascalientes,Tonelada,3473.32,7.49,300.0,0.0,7805071.04,2247.15
15,2016,5032.0,1,2,7490000,1,1,2,1,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Temporal,Aguascalientes,Tonelada,3359.06,0.67,5045.0,13.0,11255639.02,3350.83
52,2016,266.0,1,2,7490000,1,1,1,5,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Riego,Jesús María,Tonelada,3734.10,7.26,266.0,0.0,7209986.99,1930.85
60,2016,630.0,1,2,7490000,1,1,2,5,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Temporal,Jesús María,Tonelada,3345.10,0.58,630.0,0.0,1222332.99,365.41
76,2016,197.0,1,2,7490000,1,1,1,10,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Riego,El Llano,Tonelada,3509.84,7.48,197.0,0.0,5174206.13,1474.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432984,2024,85.0,4,2,7490000,190,32,1,1,200201,Jalpa,Primavera-Verano,Maíz grano,Jalpa,Zacatecas,Riego,Apozol,Tonelada,6152.94,7.75,85.0,0.0,4053249.23,658.75
432993,2024,720.0,4,2,7490000,190,32,2,1,200201,Jalpa,Primavera-Verano,Maíz grano,Jalpa,Zacatecas,Temporal,Apozol,Tonelada,6055.56,2.80,720.0,0.0,12208008.96,2016.00
433010,2024,2.0,4,1,7490000,190,32,1,19,200201,Jalpa,Otoño-Invierno,Maíz grano,Jalpa,Zacatecas,Riego,Jalpa,Tonelada,7000.00,7.50,2.0,0.0,105000.00,15.00
433021,2024,221.0,4,2,7490000,190,32,1,19,200201,Jalpa,Primavera-Verano,Maíz grano,Jalpa,Zacatecas,Riego,Jalpa,Tonelada,6165.61,7.75,221.0,0.0,10560148.53,1712.75


In [9]:
df_final[df_final['siniestrada']>0]

,anio,cosechada,idcader,idciclo,idcultivo,idddr,idestado,idmodalidad,idmunicipio,idunidadmedida,nomcader,nomcicloproductivo,nomcultivo,nomddr,nomestado,nommodalidad,nommunicipio,nomunidad,precio,rendimiento,sembrada,siniestrada,valorproduccion,volumenproduccion
15,2016,5032.0,1,2,7490000,1,1,2,1,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Temporal,Aguascalientes,Tonelada,3359.06,0.67,5045.0,13.0,1.125564e+07,3350.83
80,2016,7616.0,1,2,7490000,1,1,2,10,200201,Aguascalientes,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Temporal,El Llano,Tonelada,3372.23,0.50,8216.0,600.0,1.281683e+07,3800.70
280,2016,2123.0,3,2,7490000,1,1,2,9,200201,Pabellón,Primavera-Verano,Maíz grano,Aguascalientes,Aguascalientes,Temporal,Tepezalá,Tonelada,3409.42,0.67,3917.0,1794.0,4.849593e+06,1422.41
467,2016,0.0,3,2,7490000,2,2,2,1,200201,Ensenada,Primavera-Verano,Maíz grano,Ensenada,Baja California,Temporal,Ensenada,Tonelada,0.00,0.00,135.0,135.0,0.000000e+00,0.00
568,2016,57.0,1,2,7490000,3,2,1,2,200201,Hechicera,Primavera-Verano,Maíz grano,Río Colorado,Baja California,Riego,Mexicali,Tonelada,3750.00,8.75,61.0,4.0,1.870875e+06,498.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430187,2024,14567.0,2,2,7490000,177,30,2,108,200201,Minatitlán,Primavera-Verano,Maíz grano,Choapas,Veracruz,Temporal,Minatitlán,Tonelada,4731.59,2.20,17237.0,2670.0,1.516352e+08,32047.40
430194,2024,500.4,2,2,7490000,177,30,2,199,200201,Minatitlán,Primavera-Verano,Maíz grano,Choapas,Veracruz,Temporal,Zaragoza,Tonelada,4673.67,1.56,515.4,15.0,3.648360e+06,780.62
430227,2024,5519.4,4,2,7490000,177,30,2,209,200201,Uxpanapan,Primavera-Verano,Maíz grano,Choapas,Veracruz,Temporal,Uxpanapa,Tonelada,4739.84,1.61,5629.4,110.0,4.211931e+07,8886.23
432282,2024,300.0,1,2,7490000,186,32,2,3,200201,Tepechitlán,Primavera-Verano,Maíz grano,Tlaltenango,Zacatecas,Temporal,Atolinga,Tonelada,5766.67,6.55,340.0,40.0,1.133151e+07,1965.00


In [11]:
print(3800.70/7616.0)

0.49904149159663863


In [ ]:
cosechada: 7616.0
rendimiento: 0.50
sembrada: 8216.0
siniestrada: 600.0
volumneproduccion: 3800.70



NameError: name 'cosechada' is not defined

In [ ]:
df_final.to_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_agricolas_limpios.csv', index=False, encoding='utf-8')

In [37]:
df_final.nomcicloproductivo.count()

np.int64(55853)

In [38]:
df_final.groupby(['nomcicloproductivo']).size().reset_index(name='count')

,nomcicloproductivo,count
0,Otoño-Invierno,15213
1,Primavera-Verano,40640
